In [1]:
import os
import sys
import pandas as pd
import yaml 
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
import statsmodels.api as sm
import numpy as np
from itertools import product
import subprocess

with open("../../config.yaml.local", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)
with open("../../config.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)
sys.path.append("../python")

import globals
import data_tools as dt
import writing_tools as wt
import utils

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
RAW_DATA_PATH = LOCAL_CONFIG["RAW_DATA_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]
R_PATH = LOCAL_CONFIG["R_PATH"]

RUN_R_SCRIPTS = False

RESULTS = {}

In [2]:
qual_df = pd.read_parquet(os.path.join(DATA_PATH, "post_quality_analysis_data.parquet"))
quant_df = pd.read_parquet(os.path.join(DATA_PATH, "post_quantity_analysis_data.parquet"))
territory_df = dt.get_territory_by_day_panel()  # territory-by-day


In [3]:
# Calculate fee_diff_{jt} = fee_{jt} - fee_{j,t-1}

territory_df["fee_diff"] = territory_df.groupby("subName")["posting_fee"].diff()
territory_df["fee_chg_pos"] = (territory_df["fee_diff"] > 0).astype(int)
territory_df["fee_chg_neg"] = (territory_df["fee_diff"] < 0).astype(int)


In [4]:
# territory-by-week

twdf = territory_df.copy()
twdf["week"] = dt.as_week(twdf["date"])
twdf = twdf.groupby(["subName", "week"]).agg(
    fee_chg_pos = ("fee_chg_pos", "max"),
    fee_chg_neg = ("fee_chg_neg", "max")
).reset_index()



c:\Users\edwar\projects\sn-research\src\notebooks\../python\data_tools.py:89: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return x.dt.to_period('W-SAT').dt.start_time


In [5]:
# leads and lags for fee_chg_pos and fee_chg_neg
K = 12
for s in range(1,K+1):
    twdf[f"fee_chg_pos_F{s}"] = twdf.groupby("subName")["fee_chg_pos"].shift(-s)
    twdf[f"fee_chg_neg_F{s}"] = twdf.groupby("subName")["fee_chg_neg"].shift(-s)
    twdf[f"fee_chg_pos_L{s}"] = twdf.groupby("subName")["fee_chg_pos"].shift(s)
    twdf[f"fee_chg_neg_L{s}"] = twdf.groupby("subName")["fee_chg_neg"].shift(s)



In [6]:
# identify control units as territory-weeks with no fee change within 12 weeks

cols_to_check = [f"fee_chg_pos_F{s}" for s in range(1,13)] + [f"fee_chg_pos_L{s}" for s in range(1,13)] + [f"fee_chg_neg_F{s}" for s in range(1,13)] + [f"fee_chg_neg_L{s}" for s in range(1,13)] + ["fee_chg_pos", "fee_chg_neg"]

control_mask = twdf[cols_to_check].sum(axis=1) == 0

twdf.loc[control_mask, "group"] = "control"


In [9]:
# identify treat_pos units as territory-weeks with one positive fee change within 12 weeks but no negative fee change

pos_cols = [f"fee_chg_pos_F{s}" for s in range(1,13)] + [f"fee_chg_pos_L{s}" for s in range(1,13)] + ["fee_chg_pos"]
neg_cols = [f"fee_chg_neg_F{s}" for s in range(1,13)] + [f"fee_chg_neg_L{s}" for s in range(1,13)] + ["fee_chg_neg"]

treat_pos_mask = (twdf[pos_cols].sum(axis=1) == 1) & (twdf[neg_cols].sum(axis=1) == 0)
twdf.loc[treat_pos_mask, "group"] = "treat_pos"

In [10]:
# identify treat_neg units as territory-weeks with one negative fee change within 12 weeks but no positive fee change

treat_neg_mask = (twdf[neg_cols].sum(axis=1) == 1) & (twdf[pos_cols].sum(axis=1) == 0)
twdf.loc[treat_neg_mask, "group"] = "treat_neg"

In [11]:
twdf['group'].fillna('NA').value_counts()

group
control      3546
NA           1055
treat_pos     863
treat_neg     540
Name: count, dtype: int64

In [8]:
twdf["group"].value_counts()

group
control    3546
Name: count, dtype: int64

In [ ]:
# all posts within 12 weeks of a positive fee change

In [ ]:

qual_df = qual_df.merge(twdf, on=["subName", "week"], how="left")

# drop all posts not within 12 weeks of a fee change

cols_to_check = [f"fee_chg_pos_F{s}" for s in range(1,13)] + [f"fee_chg_pos_L{s}" for s in range(1,13)] + [f"fee_chg_neg_F{s}" for s in range(1,13)] + [f"fee_chg_neg_L{s}" for s in range(1,13)] + ["fee_chg_pos", "fee_chg_neg"]

mask =  qual_df[cols_to_check].sum(axis=1) > 0
qual_df = qual_df[mask]

# output for processing in R
qual_df.to_parquet(os.path.join(DATA_PATH, "pretrends_check_data.parquet"), index=False)
